In [12]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

In [13]:
df = pd.read_csv("creditcard.csv")

In [14]:
scaler = RobustScaler()
df["scaled_amount"] = scaler.fit_transform(df[["Amount"]])
df["scaled_time"] = scaler.fit_transform(df[["Time"]])
df = df.drop(columns=["Amount", "Time"])

In [15]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
xgb_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    use_label_encoder=False,
)
xgb_model.fit(X_train, y_train)

iso_forest = IsolationForest(
    contamination=df["Class"].mean(), random_state=42
)
iso_forest.fit(X_train)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

y_pred_iso = iso_forest.predict(X_test)
y_pred_iso = np.where(y_pred_iso == -1, 1, 0)

/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [07:51:07] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [16]:
print("=== XGBOOST EVALUATION ===")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb))
print("ROC-AUC Score:", roc_auc_score(y_test, y_proba_xgb))
print("PR-AUC Score:", average_precision_score(y_test, y_proba_xgb))

print("\n" + "=" * 40 + "\n")

print("=== ISOLATION FOREST EVALUATION ===")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_iso))
print("\nClassification Report:\n", classification_report(y_test, y_pred_iso))
print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_iso))

=== XGBOOST EVALUATION ===
Confusion Matrix:
 [[56854    10]
 [   17    81]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.89      0.83      0.86        98

    accuracy                           1.00     56962
   macro avg       0.94      0.91      0.93     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC Score: 0.9691835442674537
PR-AUC Score: 0.8727247933674632


=== ISOLATION FOREST EVALUATION ===
Confusion Matrix:
 [[56791    73]
 [   66    32]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.30      0.33      0.32        98

    accuracy                           1.00     56962
   macro avg       0.65      0.66      0.66     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC Score: 0.662623423736405
